In [1]:
import tensorflow as tf
from tensorflow.keras.layers import TextVectorization

text = open("stories.txt").read().splitlines()

vectorizer = TextVectorization(
    standardize="lower_and_strip_punctuation",
    output_mode="int",
)

vectorizer.adapt(text)

vocab = vectorizer.get_vocabulary()

print("Vocabulary Size:", len(vocab))
print(vocab[:20])

Vocabulary Size: 43
['', '[UNK]', np.str_('the'), np.str_('a'), np.str_('was'), np.str_('upon'), np.str_('time'), np.str_('there'), np.str_('village'), np.str_('once'), np.str_('knight'), np.str_('kingdom'), np.str_('fox'), np.str_('dragon'), np.str_('\ufeffonce'), np.str_('work'), np.str_('treasure'), np.str_('teaches'), np.str_('success'), np.str_('stops')]


In [3]:
tokenized = vectorizer(text)

print(tokenized[0])

tf.Tensor([14  5  3  6  7  4  3 39 10], shape=(9,), dtype=int64)


In [4]:
sequences = []

for sentence in text:
    tokens = vectorizer([sentence])[0].numpy()

    for i in range(1, len(tokens)):
        sequences.append(tokens[:i+1])

max_len = max(len(seq) for seq in sequences)

padded = tf.keras.preprocessing.sequence.pad_sequences(
    sequences,
    maxlen=max_len,
    padding='pre'
)

X = padded[:, :-1]
y = padded[:, -1]

print(X.shape)
print(y.shape)

(57, 8)
(57,)


In [5]:
import numpy as np

def positional_encoding(length, depth):

    depth = depth / 2

    positions = np.arange(length)[:, np.newaxis]
    depths = np.arange(depth)[np.newaxis, :] / depth

    angle_rates = 1 / (10000**depths)
    angle_rads = positions * angle_rates

    pos_encoding = np.concatenate(
        [np.sin(angle_rads), np.cos(angle_rads)],
        axis=-1
    )

    return tf.cast(pos_encoding, tf.float32)

In [6]:
from tensorflow.keras.layers import Layer
from tensorflow.keras.layers import Dense
from tensorflow.keras.layers import LayerNormalization
from tensorflow.keras.layers import MultiHeadAttention
from tensorflow.keras.layers import Dropout


class TransformerBlock(Layer):

    def __init__(self, embed_dim, num_heads, ff_dim):

        super().__init__()

        self.att = MultiHeadAttention(
            num_heads=num_heads,
            key_dim=embed_dim
        )

        self.ffn = tf.keras.Sequential([
            Dense(ff_dim, activation="relu"),
            Dense(embed_dim),
        ])

        self.norm1 = LayerNormalization()
        self.norm2 = LayerNormalization()

        self.dropout1 = Dropout(0.1)
        self.dropout2 = Dropout(0.1)

    def call(self, inputs):

        attn_output = self.att(
            inputs,
            inputs
        )

        attn_output = self.dropout1(attn_output)

        out1 = self.norm1(
            inputs + attn_output
        )

        ffn_output = self.ffn(out1)

        ffn_output = self.dropout2(ffn_output)

        return self.norm2(
            out1 + ffn_output
        )

In [7]:
from tensorflow.keras.layers import Embedding
from tensorflow.keras.layers import GlobalAveragePooling1D
from tensorflow.keras.layers import Input
from tensorflow.keras.models import Model

vocab_size = len(vocab)

max_len = X.shape[1]

embed_dim = 64
num_heads = 2
ff_dim = 128

inputs = Input(shape=(max_len,))

embedding_layer = Embedding(
    vocab_size,
    embed_dim
)

x = embedding_layer(inputs)

x = x + positional_encoding(
    max_len,
    embed_dim
)

x = TransformerBlock(
    embed_dim,
    num_heads,
    ff_dim
)(x)

x = GlobalAveragePooling1D()(x)

outputs = Dense(
    vocab_size,
    activation="softmax"
)(x)

model = Model(
    inputs,
    outputs
)

model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 8)              │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ embedding (Embedding)           │ (None, 8, 64)          │         2,752 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ add (Add)                       │ (None, 8, 64)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ transformer_block               │ (None, 8, 64)          │        50,048 │
│ (TransformerBlock)              │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d        │ (None, 64)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 43)             │         2,795 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 55,595 (217.17 KB)

 Trainable params: 55,595 (217.17 KB)

 Non-trainable params: 0 (0.00 B)

In [8]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

model.fit(
    X,
    y,
    epochs=100,
    batch_size=4
)

Epoch 1/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 10s 259ms/step - accuracy: 0.0351 - loss: 4.2048
Epoch 2/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.1404 - loss: 3.6522 
Epoch 3/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.1404 - loss: 3.5522 
Epoch 4/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.1404 - loss: 3.5059 
Epoch 5/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.0351 - loss: 3.4745     
Epoch 6/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.1579 - loss: 3.4231 
Epoch 7/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.1404 - loss: 3.4034 
Epoch 8/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.0877 - loss: 3.4420 
Epoch 9/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.1404 - loss: 3.3870     
Epoch 10/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - accuracy: 0.1404 - loss: 3.4125     
Epoch 11/100
15/15 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - accuracy: 0.1404 - loss: 3.4144 
Epoch 12/100
15/15 ━━━━━━━━━━━━━━━━━━━

In [13]:
import numpy as np

index_to_word = dict(
    enumerate(vocab)
)

word_to_index = {
    word:i
    for i,word in enumerate(vocab)
}


def generate(seed, num_words=10):

    result = seed

    for _ in range(num_words):

        tokenized = vectorizer([result])

        padded = tf.keras.preprocessing.sequence.pad_sequences(
            tokenized,
            maxlen=max_len,
            padding='pre'
        )

        prediction = model.predict(
            padded,
            verbose=0
        )

        next_word_id = np.argmax(
            prediction[0]
        )

        next_word = index_to_word[
            next_word_id
        ]

        result += " " + next_word

    return result

In [2]:
print(
    generate(
        "once upon",
        7
    )
)

NameError: name 'generate' is not defined